# Task Statement 4.5: Batch Processing with the Message Batches API

> Scenarios: Scenario 5 (CI) and Scenario 6 (Structured Data Extraction).

## In One Minute
- **What**: The Message Batches API runs many requests asynchronously.
- **Why**: **50% cheaper**, but results can take **up to 24 hours** and there is **no latency SLA** (Service Level Agreement — no guaranteed fast turnaround).
- **Use for**: overnight reports, weekly audits, nightly test generation. **Not for** blocking work such as a pre-merge check.
- **Handle failures**: match results by `custom_id`, resubmit only the failed ones (chunk oversized documents), and test your prompt on a small sample first.
- **Exam answer in one line**: *Batch the latency-tolerant job; keep anything a human or a merge is waiting on synchronous.*

---

## 🧒 Explain Like I'm 13

**The picture:** Post. **Express** (normal API) arrives now at full price. **Bulk freight** (Batches API) costs 50% less but may take up to 24 hours. Boxes can arrive in any order, so each one needs a **label** (`custom_id`).

**In plain steps:**
1. Is a person or a merge waiting? Use the normal call.
2. Can it wait overnight? Use a batch.
3. Match results using `custom_id`, never position.
4. If some failed, resubmit only those.
5. Check the SLA: waiting time before you submit counts too.

**Tiny example:** 30-hour SLA. Submit every 4 hours: worst case 4 + 24 = 28 hours. One big batch at the deadline can miss.

---

## 📖 Concepts Explained

### 1. Why it matters
- Half price on large, non-urgent jobs.
- Wrong fit for blocking work: a PR that waits up to a day for its review stalls the team.

### 2. How it works

**a. Shape of a batch**
```python
requests = [{"custom_id": "doc-001", "params": {"model": "claude-haiku-4-5", "max_tokens": 1024,
             "messages": [{"role": "user", "content": "..."}]}}, ...]
batch = client.messages.batches.create(requests=requests)   # real call: manual run only
# batch.processing_status: "in_progress" -> "ended"; batch.request_counts has per-type counts
```
Each result has the same `custom_id` and a `result.type` of `succeeded`, `errored`, `canceled` or `expired`. Results may come back in a different order, so always match on `custom_id`.

**b. No tools mid-batch.** A batch request is one shot. If Claude answers with `tool_use`, nothing runs the tool and sends the result back inside that request. Run the tool yourself and send a **new** request. Multi-turn tool calling does not fit a single batch request.

**c. SLA maths.** A document may arrive just after a submission. It waits up to one **submission interval**, then up to the **24-hour window**:

> `Worst Case = Interval + 24h + Margin`

For a 30-hour SLA: the interval can be at most `30 − 24 = 6` h. Leave a 2-hour margin for resubmitting failures and collecting results, and you submit **every 4 hours**. (`max_submit_interval_hours(30, 24, 2) == 4`.)

**d. Failures.** Resubmit only the `errored` or `expired` `custom_id`s. If a document was too large, split it into chunks, with new ids such as `doc-002-chunk1of3`, and map them back to `doc-002`.

**e. Refine on a sample.** Run 50 to 100 representative items first. Fix the prompt. Then submit the large batch. This avoids paying to redo 100,000 bad results.

---

## 🧪 Worked Example

### ❌ Invalid Example (Anti-Pattern / What Breaks)
*(Anti-Pattern: Batch API for Blocking PR Gate or Multi-Turn Tools)*
```python
# BROKEN: Using Message Batches API for pre-merge PR review checks
# Developer opens a PR and waits for CI check to pass before merging:
batch = client.messages.batches.create(
    requests=[{
        "custom_id": f"pr-{pr_id}",
        "params": {
            "model": "claude-sonnet-4-5",
            "messages": [{"role": "user", "content": "Review this PR diff"}]
        }
    }]
)
# Developer waits... and waits... up to 24 hours for the batch to complete!

Why it fails in production:
1. Blocking SLA violation: Pre-merge PR checks must finish in < 5 minutes. The Message Batches API has up to a 24-hour turnaround with no latency SLA.
2. No mid-batch tool calls: If Claude returns a tool_use block, the batch cannot run the tool and feed results back. Multi-turn workflows fail inside a single batch request.
```

### ✅ Valid Example (Production Pattern)
*(Production Pattern: Async Overnight Audits with SLA Scheduling & custom_id)*
```python
# CORRECT: Routing non-blocking workloads to Batches (50% discount) with custom_id correlation
# Workload: Weekly architecture audit across 40,000 files, or nightly backfill

# 1. Prepare batch requests with unique custom_id (1-64 chars: letters, digits, _, -)
requests = [
    {
        "custom_id": f"doc-audit-{file_id}",
        "params": {
            "model": "claude-haiku-4-5",
            "max_tokens": 1024,
            "messages": [{"role": "user", "content": f"Audit file: {code_content}"}]
        }
    }
    for file_id, code_content in file_manifest.items()
]

# 2. Submit batch (50% price discount, up to 100,000 requests or 256 MB per batch)
batch = client.messages.batches.create(requests=requests)

# 3. Retrieve and correlate results asynchronously by custom_id:
for result in client.messages.batches.results(batch.id):
    file_id = result.custom_id
    if result.result.type == "succeeded":
        save_finding(file_id, result.result.message.content)
    elif result.result.type in ("errored", "expired"):
        # Resubmit ONLY failed requests in the next batch window
        queue_for_retry(file_id)

# SLA Timeline for a 30-hour customer SLA:
# Target SLA (30h) - Max Batch Window (24h) - Buffer (2h) = Submit every 4 hours!
```

---

## 🔑 Key Terms

- **Anthropic Message Batches API**: An asynchronous batch processing endpoint offering a 50% discount on input and output tokens with a completion turnaround window of up to 24 hours.
- **Request Correlation (`custom_id`)**: A developer-assigned unique identifier (1–64 characters: alphanumeric, `_`, `-`) mapped 1:1 between submission requests and asynchronous response items.
- **Batch SLA Submission Interval**: The maximum scheduled window for accumulating and submitting batch jobs to ensure total turnaround satisfies customer SLA constraints (`Submit Interval = Target SLA − 24h − Buffer`).
- **Single-Turn Batch Constraint**: An architectural limitation where batches cannot pause to execute mid-request client tools, making the API suitable only for single-turn extraction or classification.
- **Selective Resubmission**: An operational error-handling standard where only `errored` or `expired` items are re-queued in subsequent batches, avoiding paying twice for successful requests.
- **Document Chunking Protocol**: Splitting an oversized payload into smaller sub-requests (such as `doc-01-chunk1of3`) to fit within token boundaries while maintaining lineage tracking.

---

## 🏗️ Architecture

```mermaid
---
config:
  htmlLabels: false
---
flowchart TD
    Work{"Is someone waiting<br>for the result?"}
    Work -->|"Yes: PR gate, chat"| Sync["Synchronous Messages API"]
    Work -->|"No: nightly, weekly"| Sample["Test prompt on a small sample"]
    Sample --> Submit["Submit batch with custom_id per item"]
    Submit --> Wait["in_progress up to 24 h"]
    Wait --> Ended["ended"]
    Ended --> Results{"result.type"}
    Results -->|"succeeded"| Save["Store by custom_id"]
    Results -->|"errored / expired"| Retry["Chunk if oversized<br>resubmit only these ids"]
    Retry --> Submit
```

---

## ⚠️ Exam Traps and Correct Patterns

### Trap 1: Batch versus synchronous

| Workload | Choice | Reason |
|---|---|---|
| Nightly audit of 50,000 documents | Batch | 50% cheaper. Nobody waits |
| Pre-merge PR check | Synchronous | Developers wait. A batch may take up to 24 hours and has no latency SLA |
| Interactive chat | Synchronous | Needs a reply now |

"Save 50%" is the bait. If a person or a merge is waiting, the answer is not batch.

### Trap 2: Assuming results come back in request order

```python
# ANTI-PATTERN
for doc, result in zip(docs, batch_results): ...   # order is not guaranteed

# PATTERN: match on custom_id
requests = [{"custom_id": f"doc-{d.id}", "params": {...}} for d in docs]
by_id = {r.custom_id: r for r in batch_results}
```

`custom_id` is 1 to 64 characters of letters, digits, `_`, `-`.

### Trap 3: Resubmitting the whole batch after partial failure

If 120 of 10,000 requests failed (for example some exceeded the context limit), read each result's type, collect the failed `custom_id`s, fix the cause (chunk the large documents), and resubmit **only those**. Rerunning all 10,000 pays twice and duplicates work.

### Trap 4: SLA arithmetic

**Exam scenario**: results are needed within 30 hours of the data arriving. Batch window is up to 24 hours.

```text
One big batch submitted at the deadline: wait up to 24h + wait for submission -> can exceed 30h.
Submit every 4 hours: worst case 4h waiting for the next submission + 24h = 28h <= 30h. OK.
```

Include the time before submission in the calculation. If the SLA is shorter than the window, do not batch.

### Trap 5: Expecting a tool loop inside a batch

A batch request is one message. It cannot pause, run your tool, and continue. If the model returns `tool_use`, run the tool yourself and submit a **new** request, or use a synchronous agent loop.

### Trap 6: Launching the full batch first

Test the prompt on a small sample synchronously. A prompt bug found after 50,000 batch requests costs the whole batch.

---

## 📝 Exam-Style Practice Questions

**Q1 (Scenario 5).** Your team wants to cut cost on two jobs: (1) a pre-merge review that blocks the PR, and (2) a weekly technical-debt report. What should you do?
- A) Batch both.
- B) Batch only the technical-debt report.
- C) Batch only the pre-merge review.
- D) Batch neither; batches are never cheaper.

**Q2 (Scenario 6).** Invoices must be processed within 30 hours of arrival, using batches with a 24-hour window. How often must you submit batches, at most?
- A) Every 24 hours.
- B) Every 12 hours.
- C) Every 6 hours at most (every 4 hours if you want a 2-hour safety margin).
- D) Only once, at the deadline.

**Q3 (Scenario 6).** In a 10,000-item batch, 40 items fail: 12 because the document was too long. What is the best recovery?
- A) Resubmit all 10,000 items.
- B) Resubmit only the 40 failed `custom_id`s, chunking the 12 long documents.
- C) Ignore the failures.
- D) Move all 10,000 to the synchronous API.

### Answers
**Q1: B.**
- A fails: the pre-merge check would wait up to 24 hours.
- C fails: it batches the blocking job and leaves the report at full price.
- D fails: batches do save 50% on latency-tolerant work.

**Q2: C.**
- A fails: a 24 h wait plus a 24 h window is 48 h.
- B fails: 12 + 24 = 36 h, over the SLA.
- D fails: one submission at the deadline cannot meet any SLA shorter than the window plus arrival wait.

**Q3: B.**
- A fails: pays again for 9,960 good results and risks duplicates.
- C fails: the data stays missing.
- D fails: it gives up the 50% saving and still does not fix the oversized documents.

---

## ✅ Self-Check Checklist
- [ ] I can state the saving (50%) and the window (up to 24 h, no latency SLA).
- [ ] I can explain why a pre-merge check must not be batched.
- [ ] I can compute a submission interval from an SLA.
- [ ] I know to resubmit only failed `custom_id`s and chunk oversized documents.
- [ ] I know a batch cannot run tools mid-request.

**Memory hook**: *Half price, day late: only if nobody is waiting.*

---

## 🛠️ Hands-on Workbook Guide

In [`ts_4_5_message_batches_api.ipynb`](ts_4_5_message_batches_api.ipynb):
1. **Step 1: Setup**: finds the repo root and adds it to the import path.
2. **Step 2: Simulator**: imports the API-shaped batch simulator and SLA helpers from `main.py`.
3. **Step 3: Simulation**: creates a batch, completes it, inspects `request_counts` and result types, and builds the retry batch.
4. **Step 4: Verification**: asserts the SLA numbers, the failed-only resubmission, and that tool calls are not executed mid-batch.

Everything runs offline; the simulator never imports `anthropic`.

---

## 🧭 Navigation & Study Links
- ⬅️ **Previous Topic**: [TS 4.4: Validation-Retry Feedback Loops](../ts_4_4_validation_retry_feedback/README.md)
- 📋 **Domain Overview**: [Domain 4: Prompt Engineering & Structured Output](../README.md)
- ➡️ **Next Topic**: [TS 4.6: Multi-Pass Review & Independent Verification](../ts_4_6_multi_pass_review/README.md)
- 🎯 **Scenario Practice**: [Master Scenario Practice](../../scenarios/MASTER_SCENARIO_PRACTICE.md)

In [ ]:
# Step 1: Environment & Setup
import sys
from pathlib import Path

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print("Environment initialized for TS 4.5 (offline, no API calls).")


### 🔍 Step 2: Batch Simulator
Import the API-shaped simulator and the SLA helpers from `main.py`. The simulator never imports `anthropic`.


In [ ]:
# Step 2: Simulator and helpers
from domain4.ts_4_5_message_batches_api.main import (
    BatchRequest,
    MessageBatchesEngine,
    chunk_failed_requests,
    is_batch_appropriate,
    max_submit_interval_hours,
    requests_needing_followup,
)

engine = MessageBatchesEngine()
print("Simulator ready.")


### 🧪 Step 3: Interactive Simulation
1. Create a batch, inspect it, complete it.
2. Resubmit only the failed ids (chunking the oversized document).
3. Show that a tool call is not executed mid-batch.


In [ ]:
# Step 3: Simulation
requests = [
    BatchRequest("doc-01", {"model": "claude-haiku-4-5", "token_estimate": 1200}),
    BatchRequest("doc-02", {"model": "claude-haiku-4-5", "token_estimate": 150_000}),
    BatchRequest("doc-03", {"model": "claude-haiku-4-5", "token_estimate": 3500}),
]
batch = engine.create_batch(requests)
print("Before:", batch["processing_status"], batch["request_counts"])
engine.complete(batch["id"])
ended = engine.retrieve(batch["id"])
print("After: ", ended["processing_status"], ended["request_counts"])

failed = engine.filter_failed_for_resubmission(batch["id"])
retry = chunk_failed_requests(requests, failed)
print("Resubmit only:", [r.custom_id for r in retry])

tool_batch = engine.submit_batch([BatchRequest("tool-1", {"tools": [{"name": "lookup_order"}]})])
followups = requests_needing_followup(engine.results(tool_batch))
print("Needs a NEW request (tool not run inside the batch):", followups)
print("Submit interval for a 30h SLA, 2h margin:", max_submit_interval_hours(30, 24, 2), "hours")


### 🔬 Step 4: Verification Assertions
We assert the SLA numbers, failed-only resubmission, and the no-tools-mid-batch behaviour.


In [ ]:
# Step 4: Verification Checks
assert ended["processing_status"] == "ended"
assert ended["request_counts"]["succeeded"] == 2 and ended["request_counts"]["errored"] == 1
assert failed == ["doc-02"]
assert [r.custom_id for r in retry] == ["doc-02-chunk1of3", "doc-02-chunk2of3", "doc-02-chunk3of3"]
assert followups == ["tool-1"]
assert max_submit_interval_hours(30, 24, 0) == 6 and max_submit_interval_hours(30, 24, 2) == 4
assert is_batch_appropriate(blocking=True, max_wait_hours=24) is False
assert is_batch_appropriate(blocking=False, max_wait_hours=24) is True

print("All verification assertions passed. Batch design verified.")
